# 🏏 IPL Match Predictor - Kaggle Notebook
## Real-time match predictions with incremental scoring

This notebook provides:
- **Match Outcome Prediction**: Predict win probabilities for any IPL match
- **Live Match Integration**: Fetches real-time data from Cricbuzz API
- **Incremental Scoring**: Confidence improves as the match progresses
- **Toss Prediction**: Predicts toss winner before match starts

In [ ]:
# Install dependencies
!pip install flask flask-cors requests -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
import warnings
import json
import os
warnings.filterwarnings('ignore')


In [ ]:

class MatchPredictor:
    """Optimized IPL match outcome predictor with Ensemble Meta-Learning"""

    def __init__(self):
        self.team_le = LabelEncoder()
        self.venue_le = LabelEncoder()
        self.scaler = StandardScaler()
        self.classes = ['A_big', 'A_small', 'B_big', 'B_small']
        self.team_stats = {}
        self.h2h_stats = {}
        self.venue_stats = {}
        self.form_stats = {}
        self.models = {}
        self.is_trained = False

    def get_rolling_stats(self, team, window=5):
        if team not in self.form_stats or len(self.form_stats[team]) < 1:
            return 0.5, 0
        recent = self.form_stats[team][-window:]
        return np.mean(recent), sum(recent[-3:]) # win rate, momentum

    def prepare_features(self, df, training=True):
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date')

        if training:
            all_teams = pd.concat([df['team_a'], df['team_b']]).unique()
            self.team_le.fit(all_teams)
            self.venue_le.fit(df['venue'].unique())

        features = []
        for idx, row in df.iterrows():
            ta, tb, venue = row['team_a'], row['team_b'], row['venue']
            
            # Team encoded
            ta_enc = self.team_le.transform([ta])[0]
            tb_enc = self.team_le.transform([tb])[0]
            v_enc = self.venue_le.transform([venue])[0]
            
            # Stats lookup
            wr_a, mom_a = self.get_rolling_stats(ta)
            wr_b, mom_b = self.get_rolling_stats(tb)
            
            h2h_key = tuple(sorted([ta, tb]))
            h2h = self.h2h_stats.get(h2h_key, {'total': 0, ta: 0, tb: 0})
            h2h_wr_a = h2h[ta] / h2h['total'] if h2h['total'] > 0 else 0.5
            
            # Update stats if training
            if training:
                winner = ta if 'A' in row['outcome'] else tb
                if ta not in self.form_stats: self.form_stats[ta] = []
                if tb not in self.form_stats: self.form_stats[tb] = []
                self.form_stats[ta].append(1 if winner == ta else 0)
                self.form_stats[tb].append(1 if winner == tb else 0)
                
                h2h['total'] += 1
                h2h[winner] += 1
                self.h2h_stats[h2h_key] = h2h

            features.append([
                ta_enc, tb_enc, v_enc, wr_a, wr_b, h2h_wr_a,
                mom_a, mom_b, wr_a - wr_b, h2h_wr_a - 0.5
            ])
        
        return np.array(features)

    def train(self, summary_csv):
        df = pd.read_csv(summary_csv)
        X = self.prepare_features(df)
        
        # Target for 4-class classification
        le = LabelEncoder()
        y = le.fit_transform(df['outcome'])
        self.classes = le.classes_
        
        self.scaler.fit(X)
        X_scaled = self.scaler.transform(X)
        
        print("[MatchPredictor] Training XGBoost + Random Forest Ensemble...")
        
        # Optimized XGBoost
        xgb = CalibratedClassifierCV(
            XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, 
                          subsample=0.8, colsample_bytree=0.8, random_state=42),
            method='sigmoid', cv=3)
            
        # Robust Random Forest
        rf = CalibratedClassifierCV(
            RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5, random_state=42),
            method='sigmoid', cv=3)
        
        xgb.fit(X_scaled, y)
        rf.fit(X_scaled, y)
        
        self.models = {'xgb': xgb, 'rf': rf}
        self.is_trained = True
        return self

    def predict_match(self, team_a, team_b, venue):
        if not self.is_trained: return {c: 0.25 for c in ['A_big', 'A_small', 'B_big', 'B_small']}
        
        ta_enc = self.team_le.transform([team_a])[0]
        tb_enc = self.team_le.transform([team_b])[0]
        v_enc = self.venue_le.transform([venue])[0]
        
        wr_a, mom_a = self.get_rolling_stats(team_a)
        wr_b, mom_b = self.get_rolling_stats(team_b)
        h2h_key = tuple(sorted([team_a, team_b]))
        h2h = self.h2h_stats.get(h2h_key, {'total': 0, team_a: 0})
        h2h_wr_a = h2h.get(team_a, 0) / h2h['total'] if h2h['total'] > 0 else 0.5
        
        feat = np.array([[ta_enc, tb_enc, v_enc, wr_a, wr_b, h2h_wr_a, mom_a, mom_b, wr_a - wr_b, h2h_wr_a - 0.5]])
        feat_scaled = self.scaler.transform(feat)
        
        # Ensemble average
        p_xgb = self.models['xgb'].predict_proba(feat_scaled)[0]
        p_rf = self.models['rf'].predict_proba(feat_scaled)[0]
        
        probs = (p_xgb * 0.6 + p_rf * 0.4)
        return dict(zip(self.classes, probs))


In [ ]:
# ============================================================================
# INCREMENTAL SCORER - Live Match Updates
# ============================================================================

class IncrementalScorer:
    """Model that improves prediction confidence as innings progresses"""

    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.classes = ['A_big', 'A_small', 'B_big', 'B_small']

    def calculate_confidence(self, overs_bowled):
        """Confidence increases with overs bowled"""
        if overs_bowled >= 20:
            return 0.95
        elif overs_bowled >= 18:
            return 0.90
        elif overs_bowled >= 15:
            return 0.80
        elif overs_bowled >= 10:
            return 0.70
        elif overs_bowled >= 6:
            return 0.55
        else:
            return 0.35

    def predict_from_live_state(self, live_state):
        """Predict based on current innings state"""
        overs = live_state.get('over', 0)
        runs = live_state.get('runs', 0)
        run_rate = live_state.get('current_run_rate', 8.0)

        confidence = self.calculate_confidence(overs)
        projected_total = run_rate * 20

        return {
            'confidence': confidence,
            'projected_total': round(projected_total, 1),
            'method': 'incremental_scorer'
        }

In [ ]:
# ============================================================================
# LIVE MATCH API CLIENT
# ============================================================================

class LiveMatchClient:
    """Client for fetching live match data and making predictions"""

    def __init__(self, predictor):
        self.predictor = predictor
        self.scorer = IncrementalScorer()
        self.base_url = "https://www.cricbuzz.com/api/cricket-match/live"

    def fetch_live_matches(self):
        """Fetch current live matches from Cricbuzz"""
        import requests
        try:
            response = requests.get(self.base_url, timeout=5)
            if response.status_code == 200:
                return response.json()
        except Exception as e:
            print(f"[LIVE] API Error: {e}")
        return None

    def parse_live_match(self, data):
        """Parse live match data to extract current match info"""
        if not data or 'matches' not in data:
            return None

        for match in data.get('matches', []):
            status = match.get('status', '').lower()
            if 'in progress' in status or 'live' in status:
                return {
                    'id': match.get('id'),
                    'status': status,
                    'team_1_name': match.get('team_1', {}).get('name', 'Unknown'),
                    'team_1_score': match.get('team_1', {}).get('score', '0/0'),
                    'team_1_overs': match.get('team_1', {}).get('overs', '0'),
                    'team_2_name': match.get('team_2', {}).get('name', 'Unknown'),
                    'team_2_score': match.get('team_2', {}).get('score', '0/0'),
                    'team_2_overs': match.get('team_2', {}).get('overs', '0'),
                    'venue': match.get('venue', 'Unknown'),
                    'series': match.get('series', 'IPL 2026'),
                    'innings': match.get('innings', 1),
                }
        return None

    def predict_live_innings(self, live_match):
        """Make prediction for live match with incremental scoring"""
        team_1 = live_match.get('team_1_name', 'Team A')
        team_2 = live_match.get('team_2_name', 'Team B')
        venue = live_match.get('venue', 'Unknown')

        # Parse score
        score_1 = live_match.get('team_1_score', '0/0')
        overs_1 = live_match.get('team_1_overs', '0')
        score_2 = live_match.get('team_2_score', '0/0')
        overs_2 = live_match.get('team_2_overs', '0')

        def parse_score(score_str):
            if not score_str or '/' not in score_str:
                return 0, 0
            parts = score_str.split('/')
            return int(parts[0]) if parts[0] else 0, int(parts[1]) if len(parts) > 1 else 0

        def parse_overs(overs_str):
            if not overs_str:
                return 0.0
            try:
                return float(overs_str)
            except:
                return 0.0

        runs_1, wickets_1 = parse_score(score_1)
        overs_1 = parse_overs(overs_1)
        runs_2, wickets_2 = parse_score(score_2)
        overs_2 = parse_overs(overs_2)

        innings = live_match.get('innings', 1)

        # Calculate overs and confidence
        if innings == 1:
            overs_bowled = overs_1
            current_runs = runs_1
        else:
            overs_bowled = overs_2
            current_runs = runs_2

        confidence = self.scorer.calculate_confidence(overs_bowled)
        run_rate = (current_runs / overs_bowled * 6) if overs_bowled > 0 else 0

        # Get base predictions
        probs = self.predictor.predict_match(team_1, team_2, venue)

        # Adjust based on current innings
        if innings == 1:
            projected_total = current_runs + ((20 - overs_bowled) * (run_rate + 2))
            if projected_total > 180:
                adjustment = 0.15
            elif projected_total > 160:
                adjustment = 0.05
            else:
                adjustment = -0.05

            base_win_prob = probs.get('A_big', 0.25) + probs.get('A_small', 0.25)
            a_total = base_win_prob + adjustment
            b_total = 1 - base_win_prob - adjustment
            probs['A_big'] = a_total * 0.6
            probs['A_small'] = a_total * 0.4
            probs['B_big'] = b_total * 0.6
            probs['B_small'] = b_total * 0.4
        else:
            target = runs_1
            required = target - current_runs + 1
            balls_left = int((20 - overs_bowled) * 6)
            required_rr = (required * 6 / balls_left) if balls_left > 0 else 999

            if required_rr > 15:
                adjustment = 0.20
            elif required_rr > 12:
                adjustment = 0.10
            elif required_rr > 10:
                adjustment = 0.0
            else:
                adjustment = -0.15

            base_win_prob = probs.get('B_big', 0.25) + probs.get('B_small', 0.25)
            new_b_prob = max(0.1, min(0.9, base_win_prob + adjustment))
            new_a_prob = 1 - new_b_prob

            probs['A_big'] = new_a_prob * 0.5
            probs['A_small'] = new_a_prob * 0.5
            probs['B_big'] = new_b_prob * 0.6
            probs['B_small'] = new_b_prob * 0.4

        # Normalize
        total = sum(probs.values())
        probs = {k: v/total for k, v in probs.items()}

        return {
            'match_info': {
                'team_1': team_1,
                'team_2': team_2,
                'venue': venue,
                'innings': innings
            },
            'live_state': {
                'team_1_score': score_1,
                'team_1_overs': overs_1,
                'team_2_score': score_2,
                'team_2_overs': overs_2,
                'run_rate': round(run_rate, 2)
            },
            'predictions': {
                'A_big': round(probs.get('A_big', 0) * 100, 1),
                'A_small': round(probs.get('A_small', 0) * 100, 1),
                'B_big': round(probs.get('B_big', 0) * 100, 1),
                'B_small': round(probs.get('B_small', 0) * 100, 1)
            },
            'most_likely': max(probs, key=probs.get),
            'confidence': round(confidence * 100, 1),
            'prediction_quality': 'improving' if overs_bowled > 10 else 'developing',
            'projected_total': round(run_rate * 20, 1)
        }

In [ ]:
# ============================================================================
# DATA LOADING & MODEL TRAINING
# ============================================================================

print("=" * 70)
print('🏏 IPL MATCH PREDICTOR - DATA LOADING')
print("=" * 70)

# Define file paths for Kaggle and Local environments
DATA_CONFIG = {
    'kaggle': {
        'match_summary': '/kaggle/input/datasett/match_summary.csv',
        'deliveries': '/kaggle/input/datasett/ipl_deliveries.csv',
        'train_ipl': '/kaggle/input/dataset/train_IPL.csv',
        'public_lb': '/kaggle/input/dataset/public_lb_matches.csv',
        'schedule': '/kaggle/input/dataset/schedule.csv',
        'sample_sub': '/kaggle/input/dataset/sample_submission.csv'
    },
    'local': {
        'match_summary': 'backend/data/match_summary.csv',
        'deliveries': 'backend/data/ipl_deliveries.csv',
        'train_ipl': 'backend/data/train_IPL.csv',
        'public_lb': 'backend/data/public_lb_matches.csv',
        'schedule': 'backend/data/schedule.csv',
        'sample_sub': 'backend/data/sample_submission.csv'
    }
}

# Detect environment
is_kaggle = os.path.exists('/kaggle/input')
env = 'kaggle' if is_kaggle else 'local'
print(f"[{env.upper()}] Environment detected")

# Load datasets
datasets = {}
for key, path_val in DATA_CONFIG[env].items():
    if os.path.exists(path_val):
        try:
            datasets[key] = pd.read_csv(path_val)
            print(f"✅ Loaded {key}: {datasets[key].shape}")
        except Exception as e:
            print(f"❌ Error loading {key}: {e}")
    else:
        print(f"⚠️ {key} not found at {path_val}")

# Initialize and train with match_summary
if 'match_summary' in datasets:
    print("\n[1] Training Match Predictor...")
    predictor = MatchPredictor()
    predictor.train(DATA_CONFIG[env]['match_summary'])
    
    print("\n[2] Initializing Live Match Client...")
    client = LiveMatchClient(predictor)
else:
    print("\n❌ CRITICAL: match_summary.csv is required for training.")


In [ ]:
# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 70)
print('SAMPLE PREDICTIONS')
print("=" * 70)

test_matches = [
    ('Sunrisers Hyderabad', 'Kolkata Knight Riders',
     'Rajiv Gandhi International Stadium, Uppal, Hyderabad'),
    ('Gujarat Titans', 'Punjab Kings',
     'Narendra Modi Stadium, Ahmedabad'),
    ('Mumbai Indians', 'Lucknow Super Giants',
     'Wankhede Stadium, Mumbai'),
]

results = []
for team_a, team_b, venue in test_matches:
    print(f"\n📊 {team_a} vs {team_b}")
    print(f"   Venue: {venue}")
    probs = predictor.predict_match(team_a, team_b, venue)
    print("   Predictions:")
    for k, v in sorted(probs.items(), key=lambda x: -x[1]):
        print(f"      {k}: {v*100:.1f}%")
    stats_a = predictor.get_team_stats(team_a)
    stats_b = predictor.get_team_stats(team_b)
    print(f"   {team_a} Win Rate: {stats_a['win_rate']*100:.1f}%")
    print(f"   {team_b} Win Rate: {stats_b['win_rate']*100:.1f}%")
    
    # Store for submission
    results.append({
        'match_id': f"{team_a[:3]}_vs_{team_b[:3]}",
        'A_big': round(probs.get('A_big', 0.25), 4),
        'A_small': round(probs.get('A_small', 0.25), 4),
        'B_big': round(probs.get('B_big', 0.25), 4),
        'B_small': round(probs.get('B_small', 0.25), 4)
    })

# Create submission DataFrame
submission_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print('SUBMISSION FILE')
print("=" * 70)
print(submission_df)
submission_df.to_csv('submission.csv', index=False)
print("\n✅ Saved to submission.csv")

In [ ]:
# ============================================================================
# LIVE MATCH PREDICTION (Fetches from Cricbuzz API)
# ============================================================================

# Uncomment to fetch live match data
# print("\n[3] Fetching live match data...")
# live_data = client.fetch_live_matches()
# 
# if live_data:
#     live_match = client.parse_live_match(live_data)
#     if live_match:
#         print(f"\n📺 Live Match: {live_match['team_1_name']} vs {live_match['team_2_name']}")
#         print(f"   Score: {live_match['team_1_score']} vs {live_match['team_2_score']}")
#         
#         # Get prediction
#         prediction = client.predict_live_innings(live_match)
#         print(f"\n📊 Live Prediction:")
#         print(f"   Confidence: {prediction['confidence']}%")
#         print(f"   Predictions: {prediction['predictions']}")
#         print(f"   Most Likely: {prediction['most_likely']}")
#     else:
#         print("No live match found")
# else:
#     print("Could not fetch live data")